# Week 1 Sales Data Exploration
This notebook demonstrates the end-to-end process of ingesting, exploring, and transforming mock sales data using Databricks.  

In this notebook:
1. Execute simple Python and SQL snippets.  
2. Upload and load the CSV into a PySpark DataFrame.
3. Perform initial DataFrame exploration (row counts, schema, summary statistics).
4. Create a temporary SQL view and run SQL queries for deeper insights.
5. Convert and clean data types, handle missing values, and add new calculated columns.  
6. Filter and order your data to generate targeted views (e.g., credit-card sales).



In [ ]:
# Display a greeting message from Python
print("Hello, Databricks!")

In [ ]:
%sql
-- Display a freeting message from SQL
SELECT "Hello from SQL!" AS message

## Data Ingestion & Exploration with PySpark

#### Load CSV Data into a PySpark DataFrame

In [ ]:
# Path to your uploaded CSV file in DBFS
file_path = "/FileStore/tables/mock_sales_data.csv"

# Read the CSV file into a DataFrame
# Infer schema and specify that the first row is the header
sales_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Display the DataFrame (shows top 20 rows by default)
sales_df.show()

# Print the schema of the DataFrame
sales_df.printSchema()

#### Basic DataFrame Exploration

In [ ]:
# Count the total number of rows
print("Total number of rows:")
print(sales_df.count())

In [ ]:
# Display the first 5 rows
print("First 5 rows:")
display(sales_df.show(5))

In [ ]:
# Generate descriptive statistics for the DataFrame
print("Descriptive statistics:")
display(sales_df.describe().show())

In [ ]:
# Display the first 5 rows of the  `TransactionID`, `Product`, `SaleAmount` columns
print("First 5 rows of `TransactionID`, `Product`, `SaleAmount` columns:")
display(sales_df.select("TransactionID", "Product", "SaleAmount").show(5))

## Basic Data Transformation & Cleansing with Spark SQL & PySpark

#### Using Spark SQL for Exploration

In [ ]:
# Create a temporary view from the DataFrame
sales_df.createOrReplaceTempView("sales_data_view")

In [ ]:
# Define a SQL query to fetch the first 10 rows from our temporary view
sql_query = "SELECT * FROM sales_data_view LIMIT 10;"

display(spark.sql(sql_query).show())

In [ ]:
# Define a SQL query to get distinct PrdoductCategory from our temporary view
sql_query = "SELECT DISTINCT ProductCategory FROM sales_data_view"

spark.sql(sql_query).show()

In [ ]:
# Define a SQL query to calculate the total sales for each product and order by total sales in descending order
sql_query = """SELECT Product, SUM(SaleAmount) as TotalSales
           FROM sales_data_view
           GROUP BY Product
           ORDER BY TotalSales DESC
        """

spark.sql(sql_query).show()

In [ ]:
# Set the CustomerID to be used in the SQL query
customer_id = "CUST085"

# Define a SQL query to select all columns for the specified CustomerID from the sales_data_view
sql_query = f"""SELECT *
           FROM sales_data_view
           WHERE CustomerID = '{customer_id}'
        """ 

display(spark.sql(sql_query))

#### Data Type Conversion (PySpark)

In [ ]:
from pyspark.sql.functions import to_date, lit, to_timestamp, col, desc, isnan, isnull
from pyspark.sql.types import DoubleType, IntegerType

In [ ]:
# Checking to_date function
display(sales_df.withColumn("TransactionDate1",to_date("TransactionDate")) \
  .limit(5))

In [ ]:
# Demonstrate converting the TransactionDate string into a Date vs. a Timestamp column
display(sales_df.withColumn("date",to_date("TransactionDate")) \
.withColumn("timestamp",to_timestamp("TransactionDate")).limit(5))

In [ ]:
# Convert the 'TransactionDate' column to a timestamp type and store it in a new column
sales_df_transformed = sales_df.withColumn("TransactionTimestamp", to_timestamp("TransactionDate", "yyyy-MM-dd HH:mm:ss"))

In [ ]:
# Check the schema of the DataFrame
sales_df_transformed.printSchema()

In [ ]:
# Display the first 5 rows of TransactionID, TransactionDate, and TransactionTimestamp columns
sales_df_transformed.select("TransactionID", "TransactionDate", "TransactionTimestamp").show(5, truncate=False)

In [ ]:
# Check IntegerType even though inferSchema already set SaleAmount to DoubleType
display(sales_df_transformed.withColumn("SaleAmount1", sales_df_transformed.SaleAmount.cast(IntegerType())).limit(5))
# sales_df_transformed.withColumn("SaleAmount1", col("SaleAmount").cast(IntegerType())).show()

#### Handling Missing Values

In [ ]:
# Count  Null elements in "PaymentMethod" column
sales_df_transformed.filter(col("PaymentMethod").isNull()).count()

In [ ]:
display(sales_df_transformed.filter(isnull(col("PaymentMethod"))))

In [ ]:
# Checking filling the null elements with "Unknown" in "PaymentMethod" columns
default_method = "Unknown"
display(sales_df_transformed.fillna(default_method, subset=["PaymentMethod"]).filter(col("TransactionID") == "TID002"))

In [ ]:
# Filling the null elements with "Unknown" in "PaymentMethod" columns
sales_df_transformed = sales_df_transformed.fillna(default_method, subset=["PaymentMethod"])

#### Creating New Columns

In [ ]:
# Check if any value in "Quantity" column less than or equal to 0
sales_df_transformed.filter(col("Quantity") <= 0).count()

In [ ]:
# Check if any value in "SaleAmount" column less than or equal to 0
sales_df_transformed.filter(col("SaleAmount") == 0).count()

In [ ]:
# Create a new column called "TotalItemCost" by multiplying "SaleAmount" and "Quantity"
sales_df_transformed = sales_df_transformed.withColumn("TotalItemCost", col("SaleAmount") * col("Quantity"))

In [ ]:
# Display the first 5 rows of the transformed sales DataFrame
display(sales_df_transformed.limit(5))

#### Filtering and Ordering

In [ ]:
# Filter the DataFrame to include only credit card sales and order by SaleAmount in descending order
credit_card_sales_df = sales_df_transformed.filter(col("PaymentMethod") == "Credit Card") \
.orderBy(col("SaleAmount").desc())

In [ ]:
credit_card_sales_df.count()

In [ ]:
display(credit_card_sales_df)

## Storing Processed Data

In [ ]:
output_path_parquet = "/FileStore/tables/innovateretail/processed_sales_parquet"
sales_df_transformed.write.mode("overwrite").parquet(output_path_parquet)
print(f"DataFrame saved to Parquet at: {output_path_parquet}")

In [ ]:
%fs ls /FileStore/tables/innovateretail/processed_sales_parquet/